# GaP: inspect a grid and its refinement primitives

This recipe is GaP-focused. It inspects a repository EGRID/INIT pair and demonstrates the lateral and vertical refinement calculations without constructing a well.

In [1]:
from pathlib import Path
import numpy as np
from src.WellClass.libs.grid_utils import GridCoarse
from src.WellClass.libs.grid_utils.LGR_grid_utils import compute_ngrd, generate_LGR_xy, generate_LGR_z

root = Path.cwd()
if not (root / 'test_data').exists():
    root = root.parent
case = root / 'test_data/examples/wildcat/model/TEMP-0'
grid = GridCoarse(str(case))
print({'NX': grid.NX, 'NY': grid.NY, 'NZ': grid.NZ, 'dx': grid.main_grd_dx, 'dy': grid.main_grd_dy})

Coarse grid dimension: nx=20, ny=20, nz=60
{'NX': 20, 'NY': 20, 'NZ': 60, 'dx': np.float32(200.0), 'dy': np.float32(200.0)}


## GaP refinement calculations

In [2]:
minimum_cell = 0.05
well_diameter = 0.31115
well_cells = compute_ngrd(well_diameter, minimum_cell)
sizes_x, sizes_y, transition = generate_LGR_xy(well_cells, minimum_cell, grid.main_grd_dx, grid.main_grd_dy)
sizes_z, numbers_z, depths_z, _ = generate_LGR_z(
    DZ_rsrv=np.array([5.0, 5.0]),
    DZ_ovb_coarse=np.array([10.0, 10.0, 10.0]),
)
print({'well_cells': well_cells, 'nx': len(sizes_x), 'ny': len(sizes_y), 'nz': len(sizes_z)})

{'well_cells': 6, 'nx': 12, 'ny': 12, 'nz': 32}


In [3]:
coarse_cells = grid.grid_init[['i', 'j', 'k', 'DX', 'DY', 'DZ']]
print('coarse grid cells:', len(coarse_cells))
assert len(sizes_x) > 0 and len(sizes_y) > 0 and len(sizes_z) > 0
assert len(coarse_cells) == grid.NX * grid.NY * grid.NZ
print('GaP grid checks passed')

coarse grid cells: 24000
GaP grid checks passed


## Visual quality control

In [ ]:
import matplotlib.pyplot as plt

permx = grid.extract_xz_slice('PERMX')
fig, ax = plt.subplots(figsize=(8, 5))
image = ax.imshow(permx, aspect='auto', origin='upper')
ax.set_title('GaP coarse-grid PERMX cross-section')
ax.set_xlabel('x cell index')
ax.set_ylabel('z cell index')
fig.colorbar(image, ax=ax, label='PERMX')
fig.tight_layout()
plt.show()